# Private full-size Ruh training repair

Uses the original **45,985,563-parameter** architecture with a new lossless V2 tokenizer. This is separate from the 3.58M pipeline validation and existing V1 epoch19 production model. Candidate promotion is **false**; no production checkpoint changes occur.

Data: Aya human Arabic/English instructions and reviewed OpenAssistant replies (Apache-2.0), Arabic Wikipedia (CC-BY-SA-3.0/GFDL). Exact revisions and source attribution are saved. Annotator IDs and private datasets are not retained.

Expected supervised-byte mix:55%Arabic dialogue,25%English dialogue,20%Arabic text. Whole conversations and duplicate prompts remain in one split.1800-second API session limit including setup;1350-second training cap; no paid capacity. Actual GPU allocation/count is recorded because Kaggle's requested accelerator can differ.


In [ ]:
import os, subprocess, sys, time
from pathlib import Path
import torch
SESSION_STARTED = time.monotonic()
assert torch.cuda.is_available(), "Kaggle GPU accelerator is required"
print({"gpu": torch.cuda.get_device_name(0), "gpu_count": torch.cuda.device_count(), "torch": torch.__version__})
# Use Kaggle's existing Torch. Only small corpus dependencies are installed.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "huggingface_hub==1.33.0", "pyarrow==25.0.1"], check=True, timeout=120)
REVISION = "708a3dd810e828d2a1836fa72fccdc2f678bca03"
REPOSITORY = Path("/kaggle/working/mizan")
subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "https://github.com/CodeWithJuber/mizan.git", str(REPOSITORY)], check=True, timeout=90)
subprocess.run(["git", "fetch", "--depth=1", "origin", REVISION], cwd=REPOSITORY, check=True, timeout=90)
subprocess.run(["git", "checkout", "--detach", REVISION], cwd=REPOSITORY, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPOSITORY, text=True).strip()
assert actual == REVISION
os.environ["HF_HOME"] = "/kaggle/working/.hf-cache"
print({"training_revision": actual, "setup_seconds": time.monotonic() - SESSION_STARTED})


In [ ]:
# All commands preserve exit codes; no shell pipelines or tail masking.
subprocess.run([sys.executable, "-m", "scripts.prepare_sequence_data", "--output-dir", "/kaggle/working/data", "--dialogues", "3000", "--sentences", "1000", "--aya-arabic", "5000", "--aya-english", "1500"], cwd=REPOSITORY, check=True, timeout=240)


In [ ]:
import json
remaining = 1720 - (time.monotonic() - SESSION_STARTED)
assert remaining > 180, "Insufficient session budget after data/setup"
training_budget = min(1350, int(remaining - 120))
subprocess.run([sys.executable, "-m", "ruh_model.train_sequence", "--data", "/kaggle/working/data/training.jsonl", "--source-metadata", "/kaggle/working/data/sources.json", "--output", "/kaggle/working/full-training-run", "--device", "cuda", "--steps", "20000", "--max-seconds", str(training_budget), "--seq-len", "384", "--batch-size", "2", "--production-architecture", "--lr", "0.0003", "--checkpoint-every-seconds", "300", "--mixing-weights", json.dumps({"ar:dialogue":0.55,"en:dialogue":0.25,"ar:text":0.20})], cwd=REPOSITORY, check=True, timeout=int(remaining))


In [ ]:
import json, hashlib, zipfile
run = Path("/kaggle/working/full-training-run")
report = json.loads((run / "evaluation.json").read_text())
report["training_revision"] = REVISION
report["session_seconds"] = time.monotonic() - SESSION_STARTED
report["accelerator"] = torch.cuda.get_device_name(0)
report["gpu_count"] = torch.cuda.device_count()
assert report["tokenizer_version"] == 2
assert report["promotion"]["approved"] is False
(run / "evaluation.json").write_text(json.dumps(report, ensure_ascii=False, indent=2))
print(json.dumps({name: report[name] for name in ["status", "parameters", "held_out_before", "held_out_after", "held_out_by_language_after", "generation_metrics", "independent_generation_metrics", "generations", "independent_generations", "promotion"]}, ensure_ascii=False, indent=2))
# Only candidate outputs; private input datasets and credentials are never packaged.
with zipfile.ZipFile("/kaggle/working/full-training-candidate.zip", "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file in sorted(run.rglob("*")):
        if file.is_file():
            archive.write(file, arcname=str(file.relative_to(run.parent)))
print({"session_seconds": time.monotonic() - SESSION_STARTED, "promotion_approved": False})
